# Step 2 — Disease genes on the interactome, and their overlap

1. Choose the score cutoff: the lowest score range with no genes supported only by weak (down-weighted) evidence sources.
2. Load the interactome (only interactions where both genes have a symbol).
3. For each disease keep the **strongly associated** genes (Open Targets `globalScore` ≥ `CUTOFF`) that are **on the network**.
4. Count shared genes between each pair of diseases and test whether the overlap is bigger than chance (hypergeometric test).

In [11]:
import math
from itertools import combinations
from pathlib import Path

import networkx as nx
import pandas as pd

DISEASE_FILES = {
    "OC":  "./inputs/diseases/ovarian cancer-MONDO_0008170.tsv",
    "HTN": "./inputs/diseases/hypertensive disorder-MONDO_0005044.tsv",
    "SBP": "./inputs/diseases/systolic blood pressure-EFO_0006335.tsv",
}

INTERACTOME_FILE = "./outputs/interactome/interactome_both_symbols.csv"

## Choose the cutoff: what kind of evidence is behind each score range?
Open Targets down-weights some sources: literature (`europepmc`), expression (`expressionAtlas`) and mouse models (`impc`)
by 0.2, and `cancerBiomarkers` by 0.5 ([source weights](https://platform-docs.opentargets.org/associations)).
A gene supported only by these weak sources cannot reach a high `globalScore`. This table shows, per score range,
the share of genes supported **only** by weak sources.
`CUTOFF` is set to the lower edge of the first score range where no disease has weak-source-only genes (in that range or any higher one).

In [12]:
# Weak = sources Open Targets down-weights (europepmc, expressionAtlas, impc: 0.2; cancerBiomarkers: 0.5).
# OTAR projects (0.5) are partner-only and not in the downloaded files.
STRONG_SOURCES = [
    "clinicalPrecedence", "gwasCredibleSets", "geneBurden", "eva", "genomicsEngland", "gene2Phenotype",
    "uniprotLiterature", "uniprotVariants", "orphanet", "clingen", "cancerGeneCensus", "intogen",
    "evaSomatic", "crisprScreen", "crispr", "reactome",
]
WEAK_SOURCES = ["europepmc", "expressionAtlas", "impc", "cancerBiomarkers"]
BINS = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 1.01]

evidence_rows = []
max_weak_score = 0.0
all_scores = []
for name, path in DISEASE_FILES.items():
    df = pd.read_csv(path, sep="\t", low_memory=False)
    all_scores.extend(df["globalScore"].dropna())
    ev = df[STRONG_SOURCES + WEAK_SOURCES].apply(pd.to_numeric, errors="coerce").fillna(0) > 0
    weak_only = ev[WEAK_SOURCES].any(axis=1) & ~ev[STRONG_SOURCES].any(axis=1)
    score_bin = pd.cut(df["globalScore"], BINS, right=False)
    for b, in_bin in weak_only.groupby(score_bin, observed=True):
        evidence_rows.append({
            "disease": name,
            "score range": str(b),
            "genes": len(in_bin),
            # weak_only is True/False per gene, so its average in this range = share of genes backed only by weak sources
            "% weak-source only": in_bin.mean() * 100,
        })
    disease_max = df.loc[weak_only, "globalScore"].max()
    max_weak_score = max(max_weak_score, disease_max)
    print(f"{name}: highest score of a weak-source-only gene = {disease_max:.3f}")
    for source in WEAK_SOURCES:
        only_this = weak_only & ev[source] & ~ev[[s for s in WEAK_SOURCES if s != source]].any(axis=1)
        print(f"    {source} only: {only_this.sum()} genes, highest score = {df.loc[only_this, 'globalScore'].max():.3f}")

# exact cutoff: the lowest actual score above the highest weak-source-only score,
# so `globalScore >= CUTOFF` keeps every gene scoring above all weak-source-only genes and none of them
CUTOFF = min(s for s in all_scores if s > max_weak_score)  # minimum Open Targets globalScore for a gene to count as a disease gene
OUT_DIR = Path("outputs") / f"cutoff_{CUTOFF:.3f}"
print(f"\nCUTOFF = {CUTOFF:.6f} (highest weak-source-only score = {max_weak_score:.6f})")

# each value = % of the disease's genes in that score range whose only evidence is from weak sources
pd.DataFrame(evidence_rows).pivot(index="score range", columns="disease", values="% weak-source only").fillna(0).round(1)

OC: highest score of a weak-source-only gene = 0.125
    europepmc only: 7802 genes, highest score = 0.120
    expressionAtlas only: 0 genes, highest score = nan
    impc only: 17 genes, highest score = 0.044
    cancerBiomarkers only: 0 genes, highest score = nan
HTN: highest score of a weak-source-only gene = 0.132
    europepmc only: 4597 genes, highest score = 0.120
    expressionAtlas only: 0 genes, highest score = nan
    impc only: 550 genes, highest score = 0.081
    cancerBiomarkers only: 0 genes, highest score = nan
SBP: highest score of a weak-source-only gene = 0.004
    europepmc only: 10 genes, highest score = 0.004
    expressionAtlas only: 0 genes, highest score = nan
    impc only: 0 genes, highest score = nan
    cancerBiomarkers only: 0 genes, highest score = nan

CUTOFF = 0.132464 (highest weak-source-only score = 0.132463)


disease,HTN,OC,SBP
score range,,,
"[0.0, 0.1)",76.3,53.0,0.5
"[0.1, 0.2)",12.4,46.5,0.0
"[0.2, 0.3)",0.0,0.0,0.0
"[0.3, 0.4)",0.0,0.0,0.0
"[0.4, 0.5)",0.0,0.0,0.0
"[0.5, 0.6)",0.0,0.0,0.0
"[0.6, 0.7)",0.0,0.0,0.0
"[0.7, 1.01)",0.0,0.0,0.0


## Load the interactome

In [13]:
interactome_df = pd.read_csv(INTERACTOME_FILE)

G = nx.from_pandas_edgelist(interactome_df, "symbol_1", "symbol_2")
network_genes = set(G.nodes())

print(f"Genes (nodes):        {G.number_of_nodes()}")
print(f"Interactions (edges): {G.number_of_edges()}")

Genes (nodes):        13270
Interactions (edges): 137863


## Load disease gene lists
Keep genes with `globalScore >= CUTOFF`, then keep only the ones that exist in the network
(genes with no known interactions can't be analysed).

In [14]:
disease_scores = {}
disease_genes = {}
summary = []

for name, path in DISEASE_FILES.items():
    df = pd.read_csv(path, sep="\t", low_memory=False)
    disease_scores[name] = df[["symbol", "globalScore"]]

    strong = set(df.loc[df["globalScore"] >= CUTOFF, "symbol"])
    on_network = strong & network_genes
    disease_genes[name] = on_network

    summary.append({
        "disease": name,
        "all genes in file": len(df),
        f"genes with score >= {CUTOFF}": len(strong),
        "on the network": len(on_network),
        "percentage": len(on_network) / len(strong) * 100,
    })

pd.DataFrame(summary)

,disease,all genes in file,genes with score >= 0.1324635127224353,on the network,percentage
0,OC,15834,1018,936,91.944990
1,HTN,10012,2064,1563,75.726744
2,SBP,4884,2601,1972,75.816993


## Gene overlap between diseases

- **shared**: number of genes in both lists
- **Jaccard** = shared / all genes in either list (0 = nothing in common, 1 = identical)
- **overlap coef.** = shared / size of the smaller list
- **expected**: how many shared genes you'd get if both lists were picked at random from the network
- **fold**: shared / expected (> 1 means more overlap than chance)
- **p-value**: hypergeometric test, probability of getting at least this many shared genes by chance

In [15]:
def hypergeom_sf(k, N, K, n):
    """P(X >= k) when drawing n genes out of N, of which K are 'successes'."""
    def log_comb(a, b):
        return math.lgamma(a + 1) - math.lgamma(b + 1) - math.lgamma(a - b + 1)

    denom = log_comb(N, n)
    return sum(
        math.exp(log_comb(K, i) + log_comb(N - K, n - i) - denom)
        for i in range(k, min(K, n) + 1)
    )


N = len(network_genes)  # background: all genes in the network
overlap_rows = []

for a, b in combinations(disease_genes, 2):
    A, B = disease_genes[a], disease_genes[b]
    shared = A & B
    expected = len(A) * len(B) / N
    overlap_rows.append({
        "pair": f"{a} - {b}",
        "size A": len(A),
        "size B": len(B),
        "shared": len(shared),
        "Jaccard": len(shared) / len(A | B),
        "overlap coef.": len(shared) / min(len(A), len(B)),
        "expected": expected,
        "fold": len(shared) / expected,
        "p-value": hypergeom_sf(len(shared), N, len(A), len(B)),
    })

overlap_df = pd.DataFrame(overlap_rows)
overlap_df

,pair,size A,size B,shared,Jaccard,overlap coef.,expected,fold,p-value
0,OC - HTN,936,1563,212,0.092698,0.226496,110.246270,1.922968,1.295844e-22
1,OC - SBP,936,1972,231,0.086291,0.246795,139.095102,1.660734,1.739473e-16
2,HTN - SBP,1563,1972,776,0.281261,0.496481,232.270987,3.340925,4.416606e-277


## Which genes are shared?

In [16]:
for a, b in combinations(disease_genes, 2):
    shared = sorted(disease_genes[a] & disease_genes[b])
    print(f"{a} & {b} ({len(shared)} genes):")
    print(", ".join(shared))
    print()

shared_all = sorted(disease_genes["OC"] & disease_genes["HTN"] & disease_genes["SBP"])
print(f"Shared by all three ({len(shared_all)} genes):")
print(", ".join(shared_all))

OC & HTN (212 genes):
ABL1, ABO, ACSL3, ACSL6, ADRB3, AGTR1, AKT3, ALDH2, ALK, ANGPT2, APC, AR, ASXL1, ASXL2, ATF1, BCAR1, BCL11B, BCL2L11, BCR, BMP5, BMPR1A, BNC2, BRCA1, BTG1, CACNA1D, CACNA1H, CAMTA1, CCDC170, CCDC88C, CCNE1, CDC73, CDH11, CDH17, CDK6, CDKN1B, CDKN2A, CDKN2B, CNTNAP2, COL2A1, COL3A1, CPS1, CREB3L1, CRTC1, CRTC3, CSF1R, CTNNB1, CUX1, CXCR4, DCC, DNAJC1, DNMT3A, EBF1, EML4, EPCAM, EPHA3, ERCC4, ERG, ESR1, FAM135B, FANCL, FAT1, FBXO11, FES, FGF5, FGF9, FGFR2, FKBP1A, FLCN, FLNA, FLT1, FLT4, FMN1, FOXP1, GNAS, GOPC, GRIN2A, GSK3B, GTF2I, HDAC7, HDAC9, HMGA1, HMGA2, HOXA11, HOXA13, HRH1, HRH2, HSPB7, IGF1R, IGF2BP2, IKBKB, INHBA, ITGB5, JAZF1, JUN, KANSL1, KAT6B, KDR, KEAP1, KIT, KLF4, KLF5, LASP1, LEF1, LIFR, MAF, MAP3K1, MECOM, MEN1, MET, MLF1, MLLT10, MT-ND6, MXI1, MYH9, NACA, NBN, NCOA1, NCOR2, NFATC2, NFE2L2, NISCH, NOTCH1, NR3C1, NRG1, NRIP1, NSD1, NT5C2, PAFAH1B2, PAX5, PDGFRA, PDGFRB, PGR, PHOX2B, PIK3R1, PIM1, POT1, PPARG, PPP2R2A, PRDM1, PRDM16, PRKAR1A, PRKCB,

## Does the result depend on the cutoff?
Repeat the overlap for several cutoffs. A robust finding should hold across them.
0.2 is the lowest cutoff worth testing: below it, genes can be linked to a disease by weak sources alone (see the cutoff check at the top).

In [17]:
sensitivity_rows = []

for cutoff in [0.2, 0.3, 0.4, 0.5]:
    genes = {
        name: set(s.loc[s["globalScore"] >= cutoff, "symbol"]) & network_genes
        for name, s in disease_scores.items()
    }
    for a, b in combinations(genes, 2):
        A, B = genes[a], genes[b]
        shared = len(A & B)
        expected = len(A) * len(B) / N
        sensitivity_rows.append({
            "cutoff": cutoff,
            "pair": f"{a} - {b}",
            "size A": len(A),
            "size B": len(B),
            "shared": shared,
            "fold": shared / expected,
            "p-value": hypergeom_sf(shared, N, len(A), len(B)),
        })

pd.DataFrame(sensitivity_rows)

,cutoff,pair,size A,size B,shared,fold,p-value
0,0.2,OC - HTN,833,1297,164,2.014326,2.270912e-19
1,0.2,OC - SBP,833,1627,179,1.752635,6.303815e-15
2,0.2,HTN - SBP,1297,1627,623,3.917702,1.265034e-258
3,0.3,OC - HTN,578,726,76,2.403367,5.767868e-13
4,0.3,OC - SBP,578,1022,90,2.021784,6.051075e-11
5,0.3,HTN - SBP,726,1022,384,6.867752,4.274591e-253
6,0.4,OC - HTN,403,308,26,2.779640,2.543029e-06
7,0.4,OC - SBP,403,492,43,2.877857,3.910038e-10
8,0.4,HTN - SBP,308,492,176,15.412311,4.976575e-179
9,0.5,OC - HTN,173,113,1,0.678807,7.744458e-01


## Save the gene lists for the next steps (localization and separation)
One gene symbol per line, same format as `MS.txt` / `PD.txt` in the Menche code.

In [18]:
gene_list_dir = OUT_DIR / "gene_lists"
gene_list_dir.mkdir(parents=True, exist_ok=True)

for name, genes in disease_genes.items():
    path = gene_list_dir / f"{name}.txt"
    with open(path, "w") as f:
        f.write("\n".join(sorted(genes)) + "\n")
    print(f"Saved {len(genes)} genes to {path}")

overlap_df.to_csv(OUT_DIR / "gene_overlap.csv", index=False)
print(f"Saved {OUT_DIR / 'gene_overlap.csv'}")

Saved 936 genes to outputs/cutoff_0.132/gene_lists/OC.txt
Saved 1563 genes to outputs/cutoff_0.132/gene_lists/HTN.txt
Saved 1972 genes to outputs/cutoff_0.132/gene_lists/SBP.txt
Saved outputs/cutoff_0.132/gene_overlap.csv


---
# Network analysis (Menche et al. 2015)

Gene overlap only counts genes that are *identical*. The network analysis also uses genes that are **close to each other** on the interactome.
We reproduce the two measures from the Menche paper (same definitions as `localization_py3.py` and `separation_py3.py`):

**1. Localization: do a disease's genes cluster together on the network?**
- **LCC (S)**: size of the largest connected component, i.e. the biggest group of disease genes that are directly linked to each other.
- **d_s**: for every disease gene, the distance (number of steps) to the closest *other* gene of the same disease, averaged.
- Both are compared with random gene sets of the same size, repeated `N_SIMS` times.
  **z-score > 1.6** (p < 0.05) for LCC means the disease genes form a real cluster (a *disease module*).

**2. Separation: how far apart are two diseases on the network?**

$$s_{AB} = \langle d_{AB} \rangle - \frac{\langle d_{AA} \rangle + \langle d_{BB} \rangle}{2}$$

- **s_AB < 0**: the two disease modules **overlap** (shared biology is likely).
- **s_AB > 0**: the modules are **separated**.
- A negative z-score means the diseases are closer than two random gene sets of the same sizes.

The distance between every pair of genes is computed once and stored in a matrix. Every gene set (real or random) then just
looks its distances up, which gives **exactly the same numbers** as Menche's pair-by-pair code but is far faster
(needed for the random repeats).

In [19]:
import random
from collections import deque

import numpy as np

N_SIMS = 10000            # number of random gene sets (Menche used 1000 for separation)
RANDOM_MODEL = "uniform" # "uniform" = Menche's full randomization; "degree" = random genes with similar degree
SEED = 42                # makes the random results reproducible

net_genes = sorted(network_genes)

## Distance between every pair of genes
Computed once and reused for all the random repeats. Takes a minute or two and about 180 MB of memory.

In [20]:
gene_index = {g: i for i, g in enumerate(net_genes)}
neighbours = [[gene_index[v] for v in G[g]] for g in net_genes]
UNREACHABLE = 255  # marks gene pairs with no path between them (real distances are far smaller)


def distances_from(source):
    """Breadth-first search: distance from one gene to every gene in the network."""
    row = np.full(len(net_genes), UNREACHABLE, dtype=np.uint8)
    dist = {source: 0}
    queue = deque([source])
    while queue:
        u = queue.popleft()
        for v in neighbours[u]:
            if v not in dist:
                dist[v] = dist[u] + 1
                queue.append(v)
    row[list(dist)] = list(dist.values())
    return row


# D[i, j] = number of steps between gene i and gene j
D = np.vstack([distances_from(s) for s in range(len(net_genes))])

print(f"Distance matrix: {D.shape[0]} x {D.shape[1]} genes, {D.nbytes / 1e6:.0f} MB")
print(f"Longest distance: {D[D < UNREACHABLE].max()} steps")

Distance matrix: 13270 x 13270 genes, 176 MB
Longest distance: 13 steps


## Helper functions

In [21]:
def lcc_size(genes):
    """Size of the largest connected component formed by `genes` (Menche: S)."""
    sub = G.subgraph(genes)
    if sub.number_of_nodes() == 0:
        return 0
    return len(max(nx.connected_components(sub), key=len))


def mean_within_distance(genes):
    """<d_s> = <d_AA>: mean distance of each gene to the closest OTHER gene of the same set."""
    a = [gene_index[g] for g in genes]
    sub = D[np.ix_(a, a)].copy()
    np.fill_diagonal(sub, UNREACHABLE)  # a gene is not its own neighbour
    nearest = sub.min(axis=1)
    return float(nearest[nearest < UNREACHABLE].mean())


def mean_between_distance(genes_a, genes_b):
    """<d_AB>: distance of every A gene to the closest B gene, and of every B gene to the closest A gene (shared genes count as 0)."""
    a = [gene_index[g] for g in genes_a]
    b = [gene_index[g] for g in genes_b]
    sub = D[np.ix_(a, b)]
    nearest = np.concatenate([sub.min(axis=1), sub.min(axis=0)])  # A -> B, then B -> A
    return float(nearest[nearest < UNREACHABLE].mean())


def separation(genes_a, genes_b):
    d_a, d_b = mean_within_distance(genes_a), mean_within_distance(genes_b)
    d_ab = mean_between_distance(genes_a, genes_b)
    return d_ab - (d_a + d_b) / 2, d_a, d_b, d_ab


# ---- random gene sets -------------------------------------------------------
# "degree" model: group genes into bins of similar degree (number of links, at least 100 genes per bin)
# and replace every disease gene with a random gene from its own bin. This controls for disease genes
# being well-studied "hub" genes with many links.
degree_bins = []
_current = []
for gene in sorted(net_genes, key=G.degree):
    _current.append(gene)
    if len(_current) >= 100 and G.degree(gene) != G.degree(_current[0]):
        degree_bins.append(_current)
        _current = []
degree_bins[-1].extend(_current)
bin_of = {g: i for i, b in enumerate(degree_bins) for g in b}


def random_like(genes, rng):
    """A random gene set of the same size (and, for the 'degree' model, similar degrees)."""
    if RANDOM_MODEL == "uniform":
        return set(rng.sample(net_genes, len(genes)))
    picked = set()
    for g in genes:
        choice = rng.choice(degree_bins[bin_of[g]])
        while choice in picked:
            choice = rng.choice(degree_bins[bin_of[g]])
        picked.add(choice)
    return picked


def z_and_p(observed, random_values, larger_is_significant):
    """z-score and empirical one-sided p-value of the observed value vs the random values."""
    random_values = np.asarray(random_values)
    z = (observed - random_values.mean()) / random_values.std()
    if larger_is_significant:
        p = (np.sum(random_values >= observed) + 1) / (len(random_values) + 1)
    else:
        p = (np.sum(random_values <= observed) + 1) / (len(random_values) + 1)
    return z, p

## 1. Localization of each disease (LCC and d_s)
Takes about 10 seconds.

In [22]:
rng = random.Random(SEED)
loc_rows = []

for name, genes in disease_genes.items():
    S = lcc_size(genes)
    d_s = mean_within_distance(genes)

    rand_S, rand_d = [], []
    for i in range(N_SIMS):
        r = random_like(genes, rng)
        rand_S.append(lcc_size(r))
        rand_d.append(mean_within_distance(r))

    z_S, p_S = z_and_p(S, rand_S, larger_is_significant=True)
    z_d, p_d = z_and_p(d_s, rand_d, larger_is_significant=False)
    loc_rows.append({
        "disease": name,
        "genes on network": len(genes),
        "LCC size S": S,
        "LCC fraction": S / len(genes),
        "random LCC (mean)": np.mean(rand_S),
        "LCC z-score": z_S,
        "LCC p-value": p_S,
        "d_s": d_s,
        "random d_s (mean)": np.mean(rand_d),
        "d_s z-score": z_d,
        "d_s p-value": p_d,
    })
    print(f"{name}: LCC = {S} (random {np.mean(rand_S):.1f}), z = {z_S:.1f}")

localization_df = pd.DataFrame(loc_rows)
localization_df

OC: LCC = 802 (random 354.0), z = 13.4
HTN: LCC = 1033 (random 810.2), z = 5.6
SBP: LCC = 1357 (random 1143.7), z = 4.9


,disease,genes on network,LCC size S,LCC fraction,random LCC (mean),LCC z-score,LCC p-value,d_s,random d_s (mean),d_s z-score,d_s p-value
0,OC,936,802,0.856838,354.0142,13.381181,0.0001,1.133547,1.578086,-14.989589,0.0001
1,HTN,1563,1033,0.660909,810.1565,5.550098,0.0001,1.325850,1.449229,-5.588099,0.0001
2,SBP,1972,1357,0.688134,1143.7450,4.875738,0.0001,1.303154,1.392123,-4.545484,0.0001


**How to read it:** LCC z-score > 1.6 means the disease genes cluster together more than random genes do,
so the disease forms a network module. Only then does it make sense to measure separation between diseases.
A negative d_s z-score means the disease genes are closer to each other than random genes are.

## 2. Separation between each pair of diseases (s_AB)
Takes a few seconds per pair with `N_SIMS = 1000`.

In [23]:
rng = random.Random(SEED)
sep_rows = []

for a, b in combinations(disease_genes, 2):
    A, B = disease_genes[a], disease_genes[b]
    s_ab, d_a, d_b, d_ab = separation(A, B)

    rand_s = []
    for i in range(N_SIMS):
        rand_s.append(separation(random_like(A, rng), random_like(B, rng))[0])
        if (i + 1) % 200 == 0:
            print(f"  {a} - {b}: {i + 1}/{N_SIMS} random repeats", end="\r")

    z, p = z_and_p(s_ab, rand_s, larger_is_significant=False)
    sep_rows.append({
        "pair": f"{a} - {b}",
        "d_AA": d_a,
        "d_BB": d_b,
        "d_AB": d_ab,
        "s_AB": s_ab,
        "random s_AB (mean)": np.mean(rand_s),
        "z-score": z,
        "p-value (closer than random)": p,
        "modules": "overlapping" if s_ab < 0 else "separated",
    })
    print(f"{a} - {b}: s_AB = {s_ab:.3f}, z = {z:.1f}              ")

separation_df = pd.DataFrame(sep_rows)
separation_df

OC - HTN: s_AB = -0.139, z = -1.2              
OC - SBP: s_AB = -0.099, z = 0.7              
HTN - SBP: s_AB = -0.563, z = -25.3              


,pair,d_AA,d_BB,d_AB,s_AB,random s_AB (mean),z-score,p-value (closer than random),modules
0,OC - HTN,1.133547,1.325850,1.090253,-0.139446,-0.118004,-1.194821,0.114789,overlapping
1,OC - SBP,1.133547,1.303154,1.119655,-0.098695,-0.109904,0.652099,0.743126,overlapping
2,HTN - SBP,1.325850,1.303154,0.751345,-0.563157,-0.184245,-25.256484,0.000100,overlapping


**How to read it:**
- **s_AB < 0**: the disease modules overlap on the network.
- **z-score < -1.6** (p < 0.05): the two diseases are significantly closer than random gene sets of the same sizes.
- HTN – SBP should come out strongly overlapping: that is the **positive control**.
- Shared genes count as distance 0, so they pull s_AB down. With the "uniform" model the random sets share almost no genes,
  so part of a negative z-score simply reflects the shared genes (see Menche SM section 2.2.1).

In [24]:
localization_df.to_csv(OUT_DIR / f"localization_{RANDOM_MODEL}.csv", index=False)
separation_df.to_csv(OUT_DIR / f"separation_{RANDOM_MODEL}.csv", index=False)
print(f"Saved {OUT_DIR / f'localization_{RANDOM_MODEL}.csv'}")
print(f"Saved {OUT_DIR / f'separation_{RANDOM_MODEL}.csv'}")

Saved outputs/cutoff_0.132/localization_uniform.csv
Saved outputs/cutoff_0.132/separation_uniform.csv
